# Explorare interactiva - preturi apartamente OLXNotebook de lucru pentru dataset-ul scrapat de pe OLX.Ruleaza celulele cu **Shift+Enter**. Modifica valorile si ruleaza din nou -asta e tot rostul unui notebook.

In [ ]:
# Ne mutam in radacina proiectului, ca sa mearga si caile, si importurile din src/import os, sysif os.path.basename(os.getcwd()) == "notebooks":    os.chdir("..")sys.path.insert(0, os.getcwd())import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snspd.set_option("display.width", 200)pd.set_option("display.max_columns", 50)sns.set_theme(style="whitegrid")print("Director de lucru:", os.getcwd())

## 1. Incarcam datele curate

In [ ]:
df = pd.read_csv("data/processed/apartamente.csv")print(f"{len(df):,} anunturi, {df.shape[1]} coloane")df.head()

In [ ]:
# Ce tipuri au coloanele si cate valori lipsa avemdf.info()

In [ ]:
# Statistici pe coloanele numericedf[["pret", "suprafata_mp", "camere", "etaj", "pret_per_mp"]].describe().round(0)

## 2. Filtrare - schimba orasul si ruleaza din nouAici incepe partea interactiva. Modifica `ORAS` si vezi ce se intampla.

In [ ]:
ORAS = "Cluj-Napoca"     # <-- schimba aici: Bucuresti, Iasi, Timisoara, Brasov...sub = df[df["oras"] == ORAS]print(f"{ORAS}: {len(sub)} anunturi")print(f"  pret median      : {sub['pret'].median():,.0f} EUR")print(f"  EUR/mp median    : {sub['pret_per_mp'].median():,.0f}")print(f"  suprafata mediana: {sub['suprafata_mp'].median():.0f} mp")sub[["pret", "suprafata_mp", "camere", "etaj", "an_constructie", "pret_per_mp"]].head(10)

In [ ]:
# Filtre combinate: apartamente de 2 camere, 50-70 mp, sub 100.000 EURfiltru = (    (df["camere"] == 2)    & (df["suprafata_mp"].between(50, 70))    & (df["pret"] < 100_000))gasite = df[filtru]print(f"{len(gasite)} anunturi corespund")gasite[["oras", "pret", "suprafata_mp", "etaj", "an_constructie", "url"]].head(10)

## 3. Clasamente

In [ ]:
# Top orase dupa pret/mp (doar cele cu cel putin 30 de anunturi)clasament = (df.groupby("oras")               .agg(anunturi=("pret", "size"),                    eur_mp=("pret_per_mp", "median"),                    pret_median=("pret", "median"))               .query("anunturi >= 30")               .sort_values("eur_mp", ascending=False))clasament.head(20).round(0)

In [ ]:
# Pivot: pret median pe judet x numar de camerepivot = df.pivot_table(values="pret", index="judet", columns="camere", aggfunc="median")pivot.sort_values(2, ascending=False).head(15).round(0)

## 4. Grafice rapide

In [ ]:
ORAS_GRAFIC = "Cluj-Napoca"     # <-- schimba si aicisub = df[df["oras"] == ORAS_GRAFIC]fig, axes = plt.subplots(1, 2, figsize=(13, 4))axes[0].scatter(sub["suprafata_mp"], sub["pret"], s=12, alpha=0.4)axes[0].set_xlabel("Suprafata (mp)"); axes[0].set_ylabel("Pret (EUR)")axes[0].set_title(f"{ORAS_GRAFIC}: pret vs suprafata")sns.boxplot(data=sub.dropna(subset=["etaj"]).assign(etaj=lambda d: d["etaj"].astype(int)),            x="etaj", y="pret_per_mp", ax=axes[1], showfliers=False)axes[1].set_title(f"{ORAS_GRAFIC}: EUR/mp pe etaj")plt.tight_layout()

## 5. Modelul in actiuneAntrenam regresia liniara si ne uitam la predictii concrete.

In [ ]:
from sklearn.linear_model import LinearRegressionfrom src.models.evaluate import incarca_date, metriciX_train, X_test, y_train, y_test = incarca_date()model = LinearRegression().fit(X_train, y_train)m = metrici(y_test, model.predict(X_test))for nume, val in m.items():    print(f"  {nume:14s} {val:,.3f}")

In [ ]:
# Cele mai bune si cele mai proaste predictii de pe setul de testpred = np.exp(model.predict(X_test))real = np.exp(y_test)rezultat = pd.DataFrame({    "real_EUR": real.values,    "prezis_EUR": pred,    "eroare_EUR": pred - real.values,    "eroare_%": 100 * (pred - real.values) / real.values,    "suprafata": np.exp(X_test["log_suprafata"]).values,    "camere": X_test["camere"].values,}).round(0)print("--- cele mai precise 5 ---")display(rezultat.reindex(rezultat["eroare_%"].abs().sort_values().index).head(5))print("--- cele mai gresite 5 ---")display(rezultat.reindex(rezultat["eroare_%"].abs().sort_values(ascending=False).index).head(5))

### Experiment: cum reactioneaza modelulLuam un apartament real din test si ii schimbam un singur feature,ca sa vedem cat conteaza. Asta e o forma simpla de analiza de sensibilitate.

In [ ]:
RAND = 0   # <-- schimba indexul ca sa testezi alt apartamentbaza = X_test.iloc[[RAND]].copy()suprafata = np.exp(baza["log_suprafata"].iloc[0])print(f"Apartament: {suprafata:.0f} mp, {baza['camere'].iloc[0]:.0f} camere, "      f"etaj {baza['etaj'].iloc[0]:.0f}")print(f"Pret real : {np.exp(y_test.iloc[RAND]):,.0f} EUR")print(f"Prezis    : {np.exp(model.predict(baza))[0]:,.0f} EUR")print()print("Daca schimbam suprafata:")for mp in [40, 55, 70, 85, 100]:    v = baza.copy()    v["log_suprafata"] = np.log(mp)    print(f"  {mp:3d} mp -> {np.exp(model.predict(v))[0]:9,.0f} EUR")print()print("Daca schimbam etajul:")for et in [0, 2, 4, 6, 8]:    v = baza.copy()    v["etaj"] = et    print(f"  etaj {et} -> {np.exp(model.predict(v))[0]:9,.0f} EUR")

> Uita-te la a doua lista: predictia abia se misca la schimbarea etajului.> Exact limitarea modelului liniar pe care am identificat-o in EDA -> nu poate invata relatia in forma de U. Reteaua neuronala ar trebui sa poata.